In [2]:
import numpy as np
import pandas as pd
import joblib
from src.metrics_manager import Metrics
from src.dataset_manager import DatasetManager

# ---------------------------------------------------------------------------
# Configuración — añadir modelos aquí cuando estén listos
# ---------------------------------------------------------------------------

PREDICTORS = {
    'NegativeBinomial': 'outputs/production/models/NegativeBinomialPiecewise_B_30w_20pc_20260509_1811.pkl',
    'DecisionTree':     'outputs/production/models/DecisionTreeModel_A_30w_10pc_20260510_1148.pkl',
    # 'RandomForest':   'outputs/production/models/RandomForestModel_X_XXw_XXpc_XXXXXXXX_XXXX.pkl',
    # 'SVR':            'outputs/production/models/SVRModel_X_XXw_XXpc_XXXXXXXX_XXXX.pkl',
    # 'XGBoost':        'outputs/production/models/XGBModel_X_XXw_XXpc_XXXXXXXX_XXXX.pkl',
}

SURVIVAL_MODELS = [
    'Cox PH (frailty)',
    'Cox PH (standard)',
    'Weibull AFT',
]

# ---------------------------------------------------------------------------
# Carga de datos
# ---------------------------------------------------------------------------

def load_test_engines() -> dict[int, pd.DataFrame]:
    """Carga los DataFrames de todos los motores de test."""
    _, m_test = DatasetManager.split_dataset()
    engines = {}
    for idx in m_test:
        df = pd.read_csv(f'data/clean/data_motor_{idx}.csv')
        df.insert(0, 'unit_number', idx)
        engines[idx] = df
    return engines

def load_train_engines() -> dict[int, pd.DataFrame]:
    """Carga los DataFrames de todos los motores de train."""
    m_train, _ = DatasetManager.split_dataset()
    engines = {}
    for idx in m_train:
        df = pd.read_csv(f'data/clean/data_motor_{idx}.csv')
        df.insert(0, 'unit_number', idx)
        engines[idx] = df
    return engines

# ---------------------------------------------------------------------------
# Evaluación por escenario
# ---------------------------------------------------------------------------

def evaluate_trajectory(
    predictor,
    engines: dict[int, pd.DataFrame],
) -> dict[str, float]:
    """Evalúa el predictor en modo Trajectory sobre todos los motores.
    
    Agrega primero a nivel de motor (engine-level) para evitar 
    pseudoreplicación, luego calcula métricas globales.
    """
    all_pred, all_true = [], []

    for motor_id, df in engines.items():
        X_df = df.drop(columns=['RUL'])
        y_pred = predictor.predict(X_df, return_mode='all')
        y_true = np.minimum(
            df['RUL'].values[predictor.window_size - 1:],
            predictor.clipping_threshold,
        )
        all_pred.append(y_pred)
        all_true.append(y_true)

    y_pred_all = np.concatenate(all_pred)
    y_true_all = np.concatenate(all_true)
    clip = predictor.clipping_threshold

    return {
        name: float(func(y_pred_all, y_true_all, clip))
        for name, func in Metrics.get_metrics().items()
    }


def evaluate_deployment(
    predictor,
    engines: dict[int, pd.DataFrame],
) -> dict[str, float]:
    """Evalúa el predictor en modo Deployment — última ventana por motor."""
    y_pred_list, y_true_list = [], []

    for motor_id, df in engines.items():
        X_df = df.drop(columns=['RUL'])
        y_pred = predictor.predict(X_df, return_mode='last')
        y_true = np.minimum(
            df['RUL'].values[-1],
            predictor.clipping_threshold,
        )
        y_pred_list.append(float(y_pred[0]))
        y_true_list.append(float(y_true))

    y_pred_arr = np.array(y_pred_list)
    y_true_arr = np.array(y_true_list)
    clip = predictor.clipping_threshold

    return {
        name: float(func(y_pred_arr, y_true_arr, clip))
        for name, func in Metrics.get_metrics().items()
    }

# ---------------------------------------------------------------------------
# Construcción de la tabla
# ---------------------------------------------------------------------------

def build_comparison_table(
    predictors: dict,
    train_engines: dict,
    test_engines: dict,
) -> pd.DataFrame:
    """Construye la tabla comparativa de todos los modelos × 4 escenarios."""

    SCENARIOS = [
        ('Train', 'Trajectory', train_engines, evaluate_trajectory),
        ('Train', 'Deployment', train_engines, evaluate_deployment),
        ('Test',  'Trajectory', test_engines,  evaluate_trajectory),
        ('Test',  'Deployment', test_engines,  evaluate_deployment),
    ]
    METRICS = ['S_score', 'C_index', 'MAE', 'RMSE']

    rows = []
    for model_name, pkl_path in predictors.items():
        print(f"  Evaluando {model_name}...")
        predictor = joblib.load(pkl_path)

        for partition, mode, engines, eval_fn in SCENARIOS:
            metrics = eval_fn(predictor, engines)
            rows.append({
                'Model':     model_name,
                'Partition': partition,
                'Mode':      mode,
                **{m: round(metrics[m], 4) for m in METRICS},
            })

    # Añadir modelos de supervivencia como filas vacías
    for surv_model in SURVIVAL_MODELS:
        for partition, mode, _, _ in SCENARIOS:
            rows.append({
                'Model':     surv_model,
                'Partition': partition,
                'Mode':      mode,
                'S_score':   None,
                'C_index':   None,
                'MAE':       None,
                'RMSE':      None,
            })

    df = pd.DataFrame(rows)
    return df


def display_table(df: pd.DataFrame) -> None:
    """Presenta la tabla en formato pivot legible."""
    # Pivot: modelos en filas, (Partition, Mode, Metric) en columnas
    pivot = df.pivot_table(
        index='Model',
        columns=['Partition', 'Mode'],
        values=['S_score', 'C_index', 'MAE', 'RMSE'],
        aggfunc='first',
    )
    # Reordenar columnas: Train/Trajectory, Train/Deployment, Test/Trajectory, Test/Deployment
    col_order = [
        ('S_score', 'Train', 'Trajectory'), ('S_score', 'Train', 'Deployment'),
        ('S_score', 'Test',  'Trajectory'), ('S_score', 'Test',  'Deployment'),
        ('C_index', 'Train', 'Trajectory'), ('C_index', 'Train', 'Deployment'),
        ('C_index', 'Test',  'Trajectory'), ('C_index', 'Test',  'Deployment'),
        ('MAE',     'Train', 'Trajectory'), ('MAE',     'Train', 'Deployment'),
        ('MAE',     'Test',  'Trajectory'), ('MAE',     'Test',  'Deployment'),
        ('RMSE',    'Train', 'Trajectory'), ('RMSE',    'Train', 'Deployment'),
        ('RMSE',    'Test',  'Trajectory'), ('RMSE',    'Test',  'Deployment'),
    ]
    col_order = [c for c in col_order if c in pivot.columns]
    pivot = pivot[col_order]
    display(pivot)

# ---------------------------------------------------------------------------
# Ejecución
# ---------------------------------------------------------------------------

print("Cargando motores...")
train_engines = load_train_engines()
test_engines  = load_test_engines()

print("\nEvaluando modelos...")
df_results = build_comparison_table(PREDICTORS, train_engines, test_engines)

print("\nTabla comparativa:")
display_table(df_results)

Cargando motores...
Training engines: 140
Test engines: 60
Training engines: 140
Test engines: 60

Evaluando modelos...
  Evaluando NegativeBinomial...
  Evaluando DecisionTree...

Tabla comparativa:


S_score                                     C_index  \
Partition             Train                  Test                 Train   
Mode             Trajectory Deployment Trajectory Deployment Trajectory   
Model                                                                     
DecisionTree         1.0261     1.0053     2.8704     0.6971     0.9407   
NegativeBinomial     1.9670     1.9825     2.4439     2.0231     0.9065   

                                                         MAE             \
Partition                         Test                 Train              
Mode             Deployment Trajectory Deployment Trajectory Deployment   
Model                                                                     
DecisionTree         0.9703     0.8981     0.9562     4.6916     4.3853   
NegativeBinomial     0.9403     0.8938     0.9444     8.2574     9.1478   

                                             RMSE                        \
Partition              Test                 Train                  Test   
Mode             Trajectory Deployment Trajectory Deployment Trajectory   
Model                                                                     
DecisionTree         7.7353     4.2842     7.6646     7.5121    12.0904   
NegativeBinomial     9.3912     9.6792    12.3680    11.5706    13.6031   

                             
Partition                    
Mode             Deployment  
Model                        
DecisionTree         6.4088  
NegativeBinomial    11.8996